# Phase 3: Pair Selection (Correlation + Cointegration)
This notebook finds candidate pairs sector-wise and ranks cointegrated pairs.

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

DATA_DIR = Path('../data')
OUT_DIR = Path('../outputs')

train = pd.read_csv(DATA_DIR / 'prices_train.csv', index_col=0, parse_dates=True)
with open(DATA_DIR / 'sectors.json', 'r') as f:
    sectors = json.load(f)

print(f'Train shape: {train.shape}')

In [ ]:
def estimate_beta(y: pd.Series, x: pd.Series) -> float:
    X = sm.add_constant(x.values)
    model = sm.OLS(y.values, X).fit()
    return float(model.params[1])

rows = []
for sector, tickers in sectors.items():
    valid = [t for t in tickers if t in train.columns]
    for a, b in combinations(valid, 2):
        corr = train[[a, b]].corr().iloc[0, 1]
        if corr < 0.80:
            continue

        s1 = train[a].dropna()
        s2 = train[b].dropna()
        joined = pd.concat([s1, s2], axis=1).dropna()
        s1j, s2j = joined.iloc[:, 0], joined.iloc[:, 1]

        coint_stat, coint_pval, _ = coint(s1j, s2j)
        beta = estimate_beta(s1j, s2j)
        spread = s1j - beta * s2j
        spread_adf_p = adfuller(spread, autolag='AIC')[1]

        rows.append({
            'sector': sector,
            'stock_a': a,
            'stock_b': b,
            'corr_train': float(corr),
            'coint_stat': float(coint_stat),
            'coint_pvalue': float(coint_pval),
            'spread_adf_pvalue': float(spread_adf_p),
            'beta_a_on_b': float(beta),
        })

pairs_df = pd.DataFrame(rows)
if pairs_df.empty:
    raise ValueError('No candidate pairs found. Try lower corr threshold.')

pairs_df['cointegrated_5pct'] = pairs_df['coint_pvalue'] < 0.05
pairs_df = pairs_df.sort_values(['cointegrated_5pct', 'coint_pvalue', 'corr_train'], ascending=[False, True, False])

display(pairs_df.head(20))
print(f'Total candidate pairs: {len(pairs_df)}')
print(f'Cointegrated (5%): {pairs_df.cointegrated_5pct.sum()}')

In [ ]:
# Select top pairs for strategy building
top_pairs = pairs_df[pairs_df['cointegrated_5pct']].groupby('sector').head(3).reset_index(drop=True)
if top_pairs.empty:
    top_pairs = pairs_df.groupby('sector').head(1).reset_index(drop=True)

display(top_pairs)

pairs_df.to_csv(DATA_DIR / 'all_candidate_pairs.csv', index=False)
top_pairs.to_csv(DATA_DIR / 'selected_pairs.csv', index=False)
print('Saved: ../data/all_candidate_pairs.csv')
print('Saved: ../data/selected_pairs.csv')

In [ ]:
# Quick visual check of top spread stationarity
import matplotlib.pyplot as plt

best = top_pairs.iloc[0]
a, b = best['stock_a'], best['stock_b']
beta = float(best['beta_a_on_b'])
spread = train[a] - beta * train[b]
z = (spread - spread.rolling(30).mean()) / spread.rolling(30).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(spread.index, spread.values)
axes[0].set_title(f'Spread: {a} - beta*{b} (beta={beta:.3f})')
axes[0].grid(alpha=0.2)

axes[1].plot(z.index, z.values)
axes[1].axhline(2, color='r', ls='--', lw=1)
axes[1].axhline(-2, color='g', ls='--', lw=1)
axes[1].axhline(0, color='black', ls='--', lw=1)
axes[1].set_title('Rolling Z-score (30D)')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUT_DIR / '04_best_pair_spread_zscore.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved: ../outputs/04_best_pair_spread_zscore.png')